In [1]:
from dotenv import load_dotenv

load_dotenv()

True

## Creating subagents

In [2]:
from langchain.tools import tool

@tool
def square_root(x: float) -> float:
    """Calculate the square root of a number"""
    return x ** 0.5

@tool
def square(x: float) -> float:
    """Calculate the square of a number"""
    return x ** 2

In [6]:
from langchain.agents import create_agent
from langchain.chat_models import init_chat_model

model = init_chat_model(
    model="gemma4",
    api_key="dummy",
    model_provider="openai",
    base_url="http://localhost:8080/v1"
)

# create subagents

subagent_1 = create_agent(
    model=model,
    tools=[square_root]
)

subagent_2 = create_agent(
    model=model,
    tools=[square]
)

## Calling subagents

In [7]:
from langchain.messages import HumanMessage

@tool
def call_subagent_1(x: float) -> float:
    """Call subagent 1 in order to calculate the square root of a number"""
    response = subagent_1.invoke({"messages": [HumanMessage(content=f"Calculate the square root of {x}")]})
    return response["messages"][-1].content

@tool
def call_subagent_2(x: float) -> float:
    """Call subagent 2 in order to calculate the square of a number"""
    response = subagent_2.invoke({"messages": [HumanMessage(content=f"Calculate the square of {x}")]})
    return response["messages"][-1].content

## Creating the main agent

main_agent = create_agent(
    model=model,
    tools=[call_subagent_1, call_subagent_2],
    system_prompt="You are a helpful assistant who can call subagents to calculate the square root or square of a number.")

## Test

In [10]:
question = "What is the square and square root of 456?"

response = main_agent.invoke({"messages": [HumanMessage(content=question)]})

In [9]:
from pprint import pprint

pprint(response)

{'messages': [HumanMessage(content='What is the square root of 456?', additional_kwargs={}, response_metadata={}, id='4a5ca5cf-bf7b-4493-8e7e-96227f67a9ab'),
              AIMessage(content='', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 18, 'prompt_tokens': 155, 'total_tokens': 173, 'completion_tokens_details': None, 'prompt_tokens_details': {'audio_tokens': None, 'cache_write_tokens': None, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gemma-4-E4B-it-UD-Q4_K_XL.gguf', 'system_fingerprint': 'b9670-02810c7aa', 'id': 'chatcmpl-Wi4HuepV4DlqmP5Wikmt7SGKig84u0aU', 'finish_reason': 'tool_calls', 'logprobs': None}, id='lc_run--01a0816c-4fee-7cd2-a804-edd0e79493ab-0', tool_calls=[{'name': 'call_subagent_1', 'args': {'x': 456}, 'id': 'SWaxyHx82BiRwCLdsHcruMvuK24TNMln', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata={'input_tokens': 155, 'output_tokens': 18, 'total_tokens': 173, 'input_token_details': {'cache_read